In [1]:
import glob
import os 
import copy
import sys
import itertools
import random
import numpy as np
import pandas as pd

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import Lasso, LassoCV
from sklearn.preprocessing import StandardScaler

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchvision import models
from torchvision.models import resnet18, ResNet18_Weights

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger

In [7]:
from lightly.loss import NTXentLoss

In [8]:
path="./"

In [ ]:
seed = 0  # 0-9

In [10]:
model="CL1"

In [11]:
latent_list=[2,4,8,16,32,64,128,256,512]

In [ ]:
proj_dim=16

# 1. Loading Data

In [9]:
df_param=pd.read_csv(path+'/data/params/param_sum.csv', index_col=0)
df_param

,DSC_Tg,"DSC_deltaH,cryst","DSC_deltaH,melt",IR1_mu1,IR1_sigma1,IR1_Int1,IR1_mu2,IR1_sigma2,IR1_Int2,IR2_mu1,...,WAXS_Int10,WAXS_mu11,WAXS_sigma11,WAXS_Int11,Process_Temp,Process_Time,Process_NucConc,Property_Deg,Property_StrainBreak,Property_Young
17_1,60.466362,-47.596012,46.480912,925.000000,8.000000,2.763751e-01,955.131085,6.280011,7.863692,640.601782,...,2660.815097,22.709726,1.452573,106.494511,75,5,0.000,0.078653,0.159800,1533.741152
18_1,61.271702,-48.692377,49.326462,925.000000,8.000000,1.026984e-02,955.259577,5.986762,7.722477,640.861463,...,2290.375137,22.817104,1.521605,94.686515,75,5,0.167,0.086981,0.088500,1595.907404
19_1,62.323349,-15.013878,51.449458,924.157782,4.768348,2.492578e+00,955.615411,5.403198,5.721598,641.002109,...,1377.228359,24.895944,5.824055,1981.239121,75,5,0.500,0.081722,0.057200,1704.734993
21_1,60.937294,-46.231835,44.462586,924.999997,0.000508,1.373156e-10,955.284399,5.796023,7.480258,640.820021,...,2342.947158,24.613239,6.064391,2356.525132,75,10,0.000,0.078401,0.079833,1549.705975
22_1,61.153538,-43.927379,50.978043,925.000000,8.000000,3.040490e-12,955.141682,6.047396,7.448746,640.502364,...,1933.092536,24.754999,5.892152,2031.016989,75,10,0.167,0.087794,0.177067,1499.181431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78_3,59.287018,-1.101862,45.724219,922.401149,5.598432,6.180660e+00,956.429912,5.037394,5.100007,640.000000,...,1042.660663,23.871538,4.654210,1827.687507,120,40,0.167,0.009879,0.034867,1853.110523
79_3,57.929401,-4.225821,42.851180,922.909254,5.136471,5.767580e+00,956.330586,5.145190,4.492747,642.000000,...,612.843045,22.823962,1.414205,133.393312,120,40,0.500,0.010791,0.039867,1812.193412
81_3,61.119339,-41.676676,47.124293,925.000000,8.000000,2.036504e-11,955.205546,6.240761,8.285110,640.686571,...,1978.966766,25.000000,5.593697,2060.534350,0,0,0.000,0.080662,0.116667,1587.940605
82_3,60.671047,-38.773534,48.770991,924.876810,7.999546,8.625324e-11,955.241218,5.947278,7.917637,640.750933,...,2269.634558,24.983987,5.620694,2226.831391,0,0,0.167,0.078363,0.099367,1442.482979


In [10]:
df_dc=df_param[['WAXS_Int1','WAXS_Int2','WAXS_Int3','WAXS_Int4','WAXS_Int5','WAXS_Int6','WAXS_Int7','WAXS_Int8','WAXS_Int9']].sum(axis=1)/df_param[['WAXS_Int1','WAXS_Int2','WAXS_Int3','WAXS_Int4','WAXS_Int5','WAXS_Int6','WAXS_Int7','WAXS_Int8','WAXS_Int9','WAXS_Int10','WAXS_Int11']].sum(axis=1)
df_dc

17_1    0.042973
18_1    0.049297
19_1    0.128264
21_1    0.022538
22_1    0.024244
          ...   
78_3    0.261020
79_3    0.595242
81_3    0.025614
82_3    0.024128
83_3    0.024049
Length: 153, dtype: float64

In [11]:
df_param=pd.read_csv(path+'/data/params/param_sum.csv',index_col=0)
df_pred=pd.concat([df_param[['Property_Deg', 'Property_StrainBreak', 'Property_Young']], df_dc], axis=1)
df_pred['Property_Deg']=df_pred['Property_Deg']*100
df_pred.columns=["Deg","Strain","Young","CrysDeg"]
df_pred["CrysDeg"]=df_pred["CrysDeg"]*100
df_pred

,Deg,Strain,Young,CrysDeg
17_1,7.8653,0.159800,1533.741152,4.297316
18_1,8.6981,0.088500,1595.907404,4.929744
19_1,8.1722,0.057200,1704.734993,12.826432
21_1,7.8401,0.079833,1549.705975,2.253836
22_1,8.7794,0.177067,1499.181431,2.424432
...,...,...,...,...
78_3,0.9879,0.034867,1853.110523,26.101993
79_3,1.0791,0.039867,1812.193412,59.524230
81_3,8.0662,0.116667,1587.940605,2.561380
82_3,7.8363,0.099367,1442.482979,2.412784


In [12]:
df_pred=df_pred.round({
    'Deg': 3,  
    'Strain': 4,  
    'Young': 0,  
    'CrysDeg': 2,  
})
df_pred

,Deg,Strain,Young,CrysDeg
17_1,7.865,0.1598,1534.0,4.30
18_1,8.698,0.0885,1596.0,4.93
19_1,8.172,0.0572,1705.0,12.83
21_1,7.840,0.0798,1550.0,2.25
22_1,8.779,0.1771,1499.0,2.42
...,...,...,...,...
78_3,0.988,0.0349,1853.0,26.10
79_3,1.079,0.0399,1812.0,59.52
81_3,8.066,0.1167,1588.0,2.56
82_3,7.836,0.0994,1442.0,2.41


In [13]:
df_IR=pd.read_csv(path+'/data/measurement/IR.csv', index_col=0)
df_WAXS=pd.read_csv(path+"/data/measurement/WAXS.csv", index_col=0)
POM_list=np.load(path+'/data/measurement/POM2.npy')

In [14]:
df_IR_norm=df_IR.div(df_IR.sum(axis=1), axis=0)
df_WAXS_norm=df_WAXS.div(df_WAXS.sum(axis=1), axis=0)
POM_list_norm=POM_list/np.max(POM_list)
df_IR.shape, df_WAXS.shape, POM_list.shape

((153, 2100), (153, 1000), (153, 496, 880))

# 2. IR

In [19]:
X_IR = df_IR_norm.values
y = df_pred.values
print (X_IR.shape, y.shape)

train_groups, test_groups = train_test_split([i for i in range(51)], test_size=0.2, random_state=seed)
print (test_groups)

train_index=train_groups+[i+51 for i in train_groups]+[i+102 for i in train_groups]
test_index=test_groups+[i+51 for i in test_groups]+[i+102 for i in test_groups]
print (test_index)

X_IR_train, y_train=X_IR[train_index].astype("float32"), y[train_index].astype("float32")
X_IR_test, y_test=X_IR[test_index].astype("float32"), y[test_index].astype("float32")

y_train_mean, y_train_std=np.mean(y_train, axis=0), np.std(y_train, axis=0)
y_train=(y_train-y_train_mean)/y_train_std
y_test=(y_test-y_train_mean)/y_train_std

X_IR_train.shape, y_train.shape, X_IR_test.shape, y_test.shape

(153, 2100) (153, 4)
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13]
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13, 94, 91, 97, 63, 75, 82, 68, 83, 54, 81, 64, 145, 142, 148, 114, 126, 133, 119, 134, 105, 132, 115]


((120, 2100), (120, 4), (33, 2100), (33, 4))

In [90]:
input_length_IR=X_IR_train.shape[1]
# =========================
# 1. Dataset
# =========================
class IRSpectrumDataset(Dataset):
    def __init__(self, X):
        X = X[:, None, :]
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]  

class IRSimCLR(pl.LightningModule):
    def __init__(self, input_length, latent=8, lr=1e-3, proj_dim=32, temperature=0.5):
        super().__init__()

        self.save_hyperparameters()

        # =========================
        # Encoder (IR CNN)
        # =========================
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Conv1d(128, 16, kernel_size=1),
            nn.ReLU(),

            nn.Flatten(),

            nn.Linear(16 * (input_length // 2), latent)
        )

        # =========================
        # projection head (SimCLR)
        # =========================
        self.projector = nn.Sequential(
            nn.Linear(latent, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

        self.loss_fn = NTXentLoss(temperature=temperature)

        self.train_loss_history = []
        self.val_loss_history = []

    # =========================
    # forward
    # =========================
    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)

        z = F.normalize(z, dim=1)

        return z

    def encode(self, x):
        h = self.encoder(x)
        z = self.projector(h)

        return h, z
    
    def augment_ir(self, x, noise_std=0.01, shift_max=2):

        # noise
        x = x + noise_std * torch.randn_like(x)

        shift = torch.randint(-shift_max, shift_max + 1, (1,), device=x.device).item()

        x = torch.roll(x, shifts=shift, dims=-1)
    
        return x

    def training_step(self, batch, batch_idx):
        x = batch  # (B,1,L)
    
        x1 = self.augment_ir(x)
        x2 = self.augment_ir(x)
    
        z1 = self(x1)
        z2 = self(x2)

        loss = self.loss_fn(z1, z2)
    
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    
        return loss

    def validation_step(self, batch, batch_idx):

        x = batch
    
        x1 = self.augment_ir(x)
        x2 = self.augment_ir(x)
    
        z1 = self(x1)
        z2 = self(x2)

        loss = self.loss_fn(z1, z2)
    
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

    def on_train_epoch_end(self):
        loss = self.trainer.callback_metrics.get("train_loss")
    
        if loss is not None:
            self.train_loss_history.append(float(loss.cpu()))

    def on_validation_epoch_end(self):
        loss = self.trainer.callback_metrics.get("val_loss")
    
        if loss is not None:
            self.val_loss_history.append(float(loss.cpu()))

In [ ]:
for latent in latent_list:
    if not os.path.exists(path + "/model/Analysis2/"+model+"/IR_"+model+"_"+str(latent)+"_"+str(seed)+".pth"):
        train_IR_dataset = IRSpectrumDataset(X_IR_train)
        test_IR_dataset  = IRSpectrumDataset(X_IR_test)
        
        train_IR_loader = DataLoader(train_IR_dataset, batch_size=16, shuffle=True)
        test_IR_loader  = DataLoader(test_IR_dataset, batch_size=16, shuffle=False)

        model_IR = IRSimCLR(input_length=input_length_IR, lr=1e-4, proj_dim=proj_dim, latent=latent)
        
        trainer_IR = pl.Trainer(
            max_epochs=500,
            enable_progress_bar=True,
            accelerator="auto",
            logger=True
        )
        
        trainer_IR.fit(model_IR, train_IR_loader, test_IR_loader)
        
        ckpt = {
                "model_state": model_IR.state_dict(),
                "train_loss_history": model_IR.train_loss_history,
                "val_loss_history": model_IR.val_loss_history,
                "hparams": model_IR.hparams
            }
        torch.save(ckpt, path + "/model/Analysis2/"+model+"/IR_"+model+"_"+str(latent)+"_"+str(seed)+".pth")

# 3. WAXS

In [72]:
X_WAXS = df_WAXS_norm.values
y = df_pred.values
print (X_WAXS.shape, y.shape)

train_groups, test_groups = train_test_split([i for i in range(51)], test_size=0.2, random_state=seed)
print (test_groups)

train_index=train_groups+[i+51 for i in train_groups]+[i+102 for i in train_groups]
test_index=test_groups+[i+51 for i in test_groups]+[i+102 for i in test_groups]
print (test_index)

X_WAXS_train, y_train=X_WAXS[train_index].astype("float32"), y[train_index].astype("float32")
X_WAXS_test, y_test=X_WAXS[test_index].astype("float32"), y[test_index].astype("float32")

y_train_mean, y_train_std=np.mean(y_train, axis=0), np.std(y_train, axis=0)
y_train=(y_train-y_train_mean)/y_train_std
y_test=(y_test-y_train_mean)/y_train_std

X_WAXS_train.shape, y_train.shape, X_WAXS_test.shape, y_test.shape

(153, 1000) (153, 4)
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13]
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13, 94, 91, 97, 63, 75, 82, 68, 83, 54, 81, 64, 145, 142, 148, 114, 126, 133, 119, 134, 105, 132, 115]


((120, 1000), (120, 4), (33, 1000), (33, 4))

In [80]:
input_length_WAXS=X_WAXS_train.shape[1]

# =========================
# 1. Dataset
# =========================
class WAXSDataset(Dataset):
    def __init__(self, X):
        X = X[:, None, :]
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]  

class WAXSSimCLR(pl.LightningModule):
    def __init__(self, input_length, latent=8, lr=1e-3, proj_dim=32, temperature=0.5):
        super().__init__()

        self.save_hyperparameters()

        # =========================
        # Encoder (IR CNN)
        # =========================
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
        
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        
            nn.MaxPool1d(kernel_size=2),
        
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Conv1d(128, 16, kernel_size=1),
            nn.ReLU(),
        
            nn.Flatten(),

            nn.Linear(16 * (input_length // 2), latent)
        )

        # =========================
        # projection head (SimCLR)
        # =========================
        self.projector = nn.Sequential(
            nn.Linear(latent, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

        self.loss_fn = NTXentLoss(temperature=temperature)

        self.train_loss_history = []
        self.val_loss_history = []

    # =========================
    # forward
    # =========================
    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)

        z = F.normalize(z, dim=1)

        return z

    def encode(self, x):
        h = self.encoder(x)
        z = self.projector(h)

        return h, z

    
    def augment_ir(self, x, noise_std=0.01, shift_max=2):

        # noise
        x = x + noise_std * torch.randn_like(x)

        shift = torch.randint(-shift_max, shift_max + 1, (1,), device=x.device).item()

        x = torch.roll(x, shifts=shift, dims=-1)
    
        return x

    def training_step(self, batch, batch_idx):
        x = batch  # (B,1,L)
    
        x1 = self.augment_ir(x)
        x2 = self.augment_ir(x)
    
        z1 = self(x1)
        z2 = self(x2)

        loss = self.loss_fn(z1, z2)
    
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    
        return loss

    def validation_step(self, batch, batch_idx):

        x = batch
    
        x1 = self.augment_ir(x)
        x2 = self.augment_ir(x)
    
        z1 = self(x1)
        z2 = self(x2)

        loss = self.loss_fn(z1, z2)
    
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

    def on_train_epoch_end(self):
        loss = self.trainer.callback_metrics.get("train_loss")
    
        if loss is not None:
            self.train_loss_history.append(float(loss.cpu()))

    def on_validation_epoch_end(self):
        loss = self.trainer.callback_metrics.get("val_loss")
    
        if loss is not None:
            self.val_loss_history.append(float(loss.cpu()))

In [ ]:
for latent in latent_list:
    if not os.path.exists(path + "/model/Analysis2/"+model+"/WAXS_"+model+"_"+str(latent)+"_"+str(seed)+".pth"):
        train_WAXS_dataset = WAXSDataset(X_WAXS_train)
        test_WAXS_dataset  = WAXSDataset(X_WAXS_test)
        
        train_WAXS_loader = DataLoader(train_WAXS_dataset, batch_size=16, shuffle=True)
        test_WAXS_loader  = DataLoader(test_WAXS_dataset, batch_size=16, shuffle=False)
        
        
        # =========================
        # 4. Training
        # =========================
        model_WAXS = WAXSSimCLR(input_length=input_length_WAXS, lr=1e-4, proj_dim=proj_dim, latent=latent)
        
        trainer_WAXS = pl.Trainer(
            max_epochs=500,
            enable_progress_bar=True,
            accelerator="auto",
            logger=True
        )
        
        trainer_WAXS.fit(model_WAXS, train_WAXS_loader, test_WAXS_loader)
        
        ckpt = {
                "model_state": model_WAXS.state_dict(),
                "train_loss_history": model_WAXS.train_loss_history,
                "val_loss_history": model_WAXS.val_loss_history,
                "hparams": model_WAXS.hparams
            }
        torch.save(ckpt, path + "/model/Analysis2/"+model+"/WAXS_"+model+"_"+str(latent)+"_"+str(seed)+".pth")

# 4. POM

In [93]:
X_POM = POM_list_norm
y = df_pred.values
print (X_POM.shape, y.shape)

train_groups, test_groups = train_test_split([i for i in range(51)], test_size=0.2, random_state=seed)
print (test_groups)

train_index=train_groups+[i+51 for i in train_groups]+[i+102 for i in train_groups]
test_index=test_groups+[i+51 for i in test_groups]+[i+102 for i in test_groups]
print (test_index)

X_POM_train, y_train=X_POM[train_index].astype("float32"), y[train_index].astype("float32")
X_POM_test, y_test=X_POM[test_index].astype("float32"), y[test_index].astype("float32")

y_train_mean, y_train_std=np.mean(y_train, axis=0), np.std(y_train, axis=0)
y_train=(y_train-y_train_mean)/y_train_std
y_test=(y_test-y_train_mean)/y_train_std

X_POM_train.shape, y_train.shape, X_POM_test.shape, y_test.shape

(153, 496, 880) (153, 4)
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13]
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13, 94, 91, 97, 63, 75, 82, 68, 83, 54, 81, 64, 145, 142, 148, 114, 126, 133, 119, 134, 105, 132, 115]


((120, 496, 880), (120, 4), (33, 496, 880), (33, 4))

In [97]:
input_length1, input_length2=X_POM_train.shape[1], X_POM_train.shape[2]
# =========================
# 1. Dataset
# =========================
class POMDataset(Dataset):
    def __init__(self, X):
        # (N, length) → (N, 1, length)
        X = X[:, None, :]
        
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 4)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]


# =========================
# 2. Lightning Model
# =========================
class POMSimCLR(pl.LightningModule):

    def __init__(self, input_height, input_width, latent=8, proj_dim=proj_dim, lr=1e-3, temperature=0.5):
        super().__init__()

        self.save_hyperparameters()

        # =========================
        # Encoder (2D CNN)
        # =========================
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        
            nn.Conv2d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        
            nn.Conv2d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(128, latent, kernel_size=1),
            nn.ReLU(),
            
            nn.AdaptiveAvgPool2d((1,1)),
        
            nn.Flatten()
        )

        # =========================
        # projector head
        # =========================
        self.projector = nn.Sequential(
            nn.Linear(latent, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

        self.temperature = temperature
        self.loss_fn = NTXentLoss(temperature=temperature)

        self.train_loss_history = []
        self.val_loss_history = []

    # =========================
    # forward
    # =========================
    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        z = F.normalize(z, dim=1)
        return z

    def augment_pom(self, x):
        # noise
        x = x + 0.01 * torch.randn_like(x)
    
        # random flip
        if torch.rand(1).item() > 0.5:
            x = torch.flip(x, dims=[-1])
    
        if torch.rand(1).item() > 0.5:
            x = torch.flip(x, dims=[-2])
    
        # small translation
        shift_x = torch.randint(-2, 3, (1,)).item()
        shift_y = torch.randint(-2, 3, (1,)).item()
    
        x = torch.roll(x, shifts=(shift_x, shift_y), dims=(-2, -1))
    
        return x

    def encode(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, z
    
    def training_step(self, batch, batch_idx):
        x = batch
    
        x1 = self.augment_pom(x)
        x2 = self.augment_pom(x)
    
        z1 = self(x1)
        z2 = self(x2)
    
        loss = self.loss_fn(z1, z2)
    
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    
        return loss

    def validation_step(self, batch, batch_idx):
        x = batch
    
        x1 = self.augment_pom(x)
        x2 = self.augment_pom(x)
    
        z1 = self(x1)
        z2 = self(x2)
    
        loss = self.loss_fn(z1, z2)
    
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
    
    
    def on_train_epoch_end(self):
        loss = self.trainer.callback_metrics.get("train_loss")
        if loss is not None:
            self.train_loss_history.append(float(loss.cpu()))
    
    def on_validation_epoch_end(self):
        loss = self.trainer.callback_metrics.get("val_loss")
        if loss is not None:
            self.val_loss_history.append(float(loss.cpu()))

In [ ]:
for latent in latent_list:
    if not os.path.exists(path + "/model/Analysis2/"+model+"/POM_"+model+"_"+str(latent)+"_"+str(seed)+".pth"):
        train_POM_dataset = POMDataset(X_POM_train)
        test_POM_dataset  = POMDataset(X_POM_test)
        
        train_POM_loader = DataLoader(train_POM_dataset, batch_size=16, shuffle=True)
        test_POM_loader  = DataLoader(test_POM_dataset, batch_size=16, shuffle=False)
        
        model_POM = POMSimCLR(input_height=input_length1, input_width=input_length2, lr=1e-4,
                              proj_dim=proj_dim, latent=latent)
        
        trainer_POM = pl.Trainer(
            max_epochs=500,
            enable_progress_bar=True,
            accelerator="auto",
            logger=True
        )
        
        trainer_POM.fit(model_POM, train_POM_loader, test_POM_loader)
        
        ckpt = {
                "model_state": model_POM.state_dict(),
                "train_loss_history": model_POM.train_loss_history,
                "val_loss_history": model_POM.val_loss_history,
                "hparams": model_POM.hparams
            }
        torch.save(ckpt, path + "/model/Analysis2/"+model+"/POM_"+model+"_"+str(latent)+"_"+str(seed)+".pth")

# 5. Latent Embedding Extraction

In [98]:
class AllDataset(Dataset):
    def __init__(self, X_IR, X_WAXS, X_POM, y):
        # (N, length) → (N, 1, length)
        X_IR = X_IR[:, None, :]
        X_WAXS = X_WAXS[:, None, :]
        X_POM = X_POM[:, None, :]
        
        self.X1 = torch.tensor(X_IR, dtype=torch.float32)
        self.X2 = torch.tensor(X_WAXS, dtype=torch.float32)
        self.X3 = torch.tensor(X_POM, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 4)

    def __len__(self):
        return len(self.X1)

    def __getitem__(self, idx):
        return self.X1[idx], self.X2[idx], self.X3[idx], self.y[idx]

In [105]:
for latent in latent_list:
    model_IR2=IRSimCLR(input_length=input_length_IR, latent=latent, proj_dim=proj_dim)
    model_IR2.load_state_dict(torch.load(path + "/model/Analysis2/"+model+"/IR_"+model+"_"+str(latent)+"_"+str(seed)+".pth", map_location="cpu",
                                         weights_only=False)["model_state"])
    model_WAXS2=WAXSSimCLR(input_length=input_length_WAXS, lr=1e-4,
                           latent=latent, proj_dim=proj_dim)
    model_WAXS2.load_state_dict(torch.load(path + "/model/Analysis2/"+model+"/WAXS_"+model+"_"+str(latent)+"_"+str(seed)+".pth", map_location="cpu",
                                         weights_only=False)["model_state"])
    model_POM2=POMSimCLR(input_height=input_length1, input_width=input_length2, lr=1e-4,
                         latent=latent, proj_dim=16)
    model_POM2.load_state_dict(torch.load(path + "/model/Analysis2/"+model+"/POM_"+model+"_"+str(latent)+"_"+str(seed)+".pth", map_location="cpu",
                                         weights_only=False)["model_state"])
    model_IR2.eval()
    model_WAXS2.eval()
    model_POM2.eval()

    train_All_dataset = AllDataset(X_IR_train, X_WAXS_train, X_POM_train, y_train)
    test_All_dataset  = AllDataset(X_IR_test, X_WAXS_test, X_POM_test, y_test)
    
    train_All_loader = DataLoader(train_All_dataset, batch_size=1, shuffle=False)
    test_All_loader  = DataLoader(test_All_dataset, batch_size=1, shuffle=False)

    mid_IR_train, mid_WAXS_train, mid_POM_train, y_train_real = [], [], [], []
    mid_IR_train2, mid_WAXS_train2, mid_POM_train2 = [], [], []
    mid_IR_test,  mid_WAXS_test,  mid_POM_test, y_test_real  = [], [], [], [] 
    mid_IR_test2,  mid_WAXS_test2,  mid_POM_test2  = [], [], [] 
    
    with torch.no_grad():
        for x_IR, x_WAXS, x_POM, y in train_All_loader:
            latent_IR, z_IR = model_IR2.encode(x_IR)
            latent_WAXS, z_WAXS = model_WAXS2.encode(x_WAXS)
            latent_POM, z_POM = model_POM2.encode(x_POM)
            mid_IR_train.append(latent_IR.cpu().numpy()[0])
            mid_WAXS_train.append(latent_WAXS.cpu().numpy()[0])
            mid_POM_train.append(latent_POM.cpu().numpy()[0])
            mid_IR_train2.append(z_IR.cpu().numpy()[0])
            mid_WAXS_train2.append(z_WAXS.cpu().numpy()[0])
            mid_POM_train2.append(z_POM.cpu().numpy()[0])
            y_train_real.append(y.cpu().numpy()[0])
    
        for x_IR, x_WAXS, x_POM, y in test_All_loader:
            latent_IR, z_IR = model_IR2.encode(x_IR)
            latent_WAXS, z_WAXS = model_WAXS2.encode(x_WAXS)
            latent_POM, z_POM = model_POM2.encode(x_POM)
            mid_IR_test.append(latent_IR.cpu().numpy()[0])
            mid_WAXS_test.append(latent_WAXS.cpu().numpy()[0])
            mid_POM_test.append(latent_POM.cpu().numpy()[0])
            mid_IR_test2.append(z_IR.cpu().numpy()[0])
            mid_WAXS_test2.append(z_WAXS.cpu().numpy()[0])
            mid_POM_test2.append(z_POM.cpu().numpy()[0])
            y_test_real.append(y.cpu().numpy()[0])
    
    X_mid_train = pd.concat([pd.DataFrame(mid_IR_train), pd.DataFrame(mid_WAXS_train), pd.DataFrame(mid_POM_train)], axis=1)
    X_mid_test  = pd.concat([pd.DataFrame(mid_IR_test),  pd.DataFrame(mid_WAXS_test),  pd.DataFrame(mid_POM_test)],  axis=1)
    X_mid_train2 = pd.concat([pd.DataFrame(mid_IR_train2), pd.DataFrame(mid_WAXS_train2), pd.DataFrame(mid_POM_train2)], axis=1)
    X_mid_test2  = pd.concat([pd.DataFrame(mid_IR_test2),  pd.DataFrame(mid_WAXS_test2),  pd.DataFrame(mid_POM_test2)],  axis=1)

    y_train_real=np.array(y_train_real)
    y_test_real =np.array(y_test_real)

    X_mid_train.to_csv(path+'/data/mid/'+model+'/mid_train_'+model+'_'+str(latent)+"_"+str(seed)+'.csv')
    X_mid_test.to_csv(path+'/data/mid/'+model+'/mid_test_'+model+'_'+str(latent)+"_"+str(seed)+'.csv')
    X_mid_train2.to_csv(path+'/data/mid/'+model+'/mid_train_'+model+'_'+str(latent)+"_"+str(seed)+'-2.csv')
    X_mid_test2.to_csv(path+'/data/mid/'+model+'/mid_test_'+model+'_'+str(latent)+"_"+str(seed)+'-2.csv')
    pd.DataFrame(y_train_real).to_csv(path+'/data/mid/'+model+'/y_train_'+model+'_'+str(latent)+"_"+str(seed)+'.csv')
    pd.DataFrame(y_test_real).to_csv(path+'/data/mid/'+model+'/y_test_'+model+'_'+str(latent)+"_"+str(seed)+'.csv')

In [106]:
y_name=["Deg","Strain", "Young", "CrysDeg"]
indicator_name=["Deg,train", "Deg,test","Strain,train", "Strain,test","Young,train", "Young,test","CryDeg,train", "CryDeg,test"]

In [107]:
df_result_sum=pd.DataFrame(columns=indicator_name)
df_result_sum2=pd.DataFrame(columns=indicator_name)

,"Deg,train","Deg,test","Strain,train","Strain,test","Young,train","Young,test","CryDeg,train","CryDeg,test"


# 6. LASSO 

In [163]:
def reg(kind="all", method="Lasso", df_result_sum=df_result_sum, hosei=True):
    for latent in latent_list:
        result=[]
        X_mid_train=pd.read_csv(path+'/data/mid/'+model+'/mid_train_'+model+'_'+str(latent)+"_"+str(seed)+'.csv').iloc[:,1:]
        X_mid_test =pd.read_csv(path+'/data/mid/'+model+'/mid_test_'+model+'_'+str(latent)+"_"+str(seed)+'.csv').iloc[:,1:]
        if hosei:
            eps = np.finfo(np.float64).eps
            ave=copy.copy(np.repeat([X_mid_train.iloc[:,:latent].mean().mean()+eps,X_mid_train.iloc[:,latent:latent*2].mean().mean()+eps,X_mid_train.iloc[:,latent*2:].mean().mean()+eps], latent))
            X_mid_train=X_mid_train/ave
            X_mid_test=X_mid_test/ave
        y_train_real=pd.read_csv(path+'/data/mid/'+model+'/y_train_'+model+'_'+str(latent)+"_"+str(seed)+'.csv').iloc[:,1:]
        y_test_real =pd.read_csv(path+'/data/mid/'+model+'/y_test_'+model+'_'+str(latent)+"_"+str(seed)+'.csv').iloc[:,1:]
        if kind=="all":
            X_mid_train, X_mid_test=X_mid_train, X_mid_test
        elif kind=="IR":
            X_mid_train, X_mid_test=X_mid_train.iloc[:,:latent], X_mid_test.iloc[:,:latent]
        elif kind=="WAXS":
            X_mid_train, X_mid_test=X_mid_train.iloc[:,latent:latent*2], X_mid_test.iloc[:,latent:latent*2]
        elif kind=="POM":
            X_mid_train, X_mid_test=X_mid_train.iloc[:,latent*2:], X_mid_test.iloc[:,latent*2:]

        print (X_mid_train.shape, X_mid_test.shape, y_train_real.shape, y_test_real.shape)
            
        non_zero_coef=[]
        for i in range(4):
            import warnings
            from sklearn.exceptions import ConvergenceWarning
            warnings.filterwarnings("ignore", category=ConvergenceWarning)
            lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
            lasso_cv.fit(X_mid_train, y_train_real[str(i)])
            lasso = Lasso(alpha=lasso_cv.alpha_)
            lasso.fit(X_mid_train, y_train_real[str(i)])
            y_train_pred=lasso.predict(X_mid_train)
            y_test_pred=lasso.predict(X_mid_test)
            non_zero_coef.append(int(np.sum(lasso.coef_ != 0)))
            if i==3:
                print (non_zero_coef)
              
            result.append(r2_score(y_train_real[str(i)], y_train_pred))
            result.append(r2_score(y_test_real[str(i)], y_test_pred))
        df_result=pd.DataFrame(result,index=indicator_name, columns=[kind+','+method+','+str(latent)])
        df_result_sum=pd.concat([df_result_sum, df_result.T])
        
    return df_result_sum

In [ ]:
def reg2(kind="all", method="Lasso", df_result_sum=df_result_sum2, proj_dim=16, hosei=True):
    for latent in latent_list:
        result=[]
        X_mid_train=pd.read_csv(path+'/data/mid/'+model+'/mid_train_'+model+'_'+str(latent)+"_"+str(seed)+'-2.csv').iloc[:,1:]
        X_mid_test =pd.read_csv(path+'/data/mid/'+model+'/mid_test_'+model+'_'+str(latent)+"_"+str(seed)+'-2.csv').iloc[:,1:]
        if hosei:
            eps = np.finfo(np.float64).eps
            ave=copy.copy(np.repeat([X_mid_train.iloc[:,:proj_dim].mean().mean()+eps,X_mid_train.iloc[:,proj_dim:proj_dim*2].mean().mean()+eps,X_mid_train.iloc[:,proj_dim*2:].mean().mean()+eps], proj_dim))
            X_mid_train=X_mid_train/ave
            X_mid_test=X_mid_test/ave
        y_train_real=pd.read_csv(path+'/data/mid/'+model+'/y_train_'+model+'_'+str(latent)+"_"+str(seed)+'.csv').iloc[:,1:]
        y_test_real =pd.read_csv(path+'/data/mid/'+model+'/y_test_'+model+'_'+str(latent)+"_"+str(seed)+'.csv').iloc[:,1:]
        if kind=="all":
            X_mid_train, X_mid_test=X_mid_train, X_mid_test
        elif kind=="IR":
            X_mid_train, X_mid_test=X_mid_train.iloc[:,:proj_dim], X_mid_test.iloc[:,:proj_dim]
        elif kind=="WAXS":
            X_mid_train, X_mid_test=X_mid_train.iloc[:,proj_dim:proj_dim*2], X_mid_test.iloc[:,proj_dim:proj_dim*2]
        elif kind=="POM":
            X_mid_train, X_mid_test=X_mid_train.iloc[:,proj_dim*2:], X_mid_test.iloc[:,proj_dim*2:]
        print (X_mid_train.shape, X_mid_test.shape, y_train_real.shape, y_test_real.shape)
            
        non_zero_coef=[]
        for i in range(4):
            import warnings
            from sklearn.exceptions import ConvergenceWarning
            warnings.filterwarnings("ignore", category=ConvergenceWarning)
            lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
            lasso_cv.fit(X_mid_train, y_train_real[str(i)])
            lasso = Lasso(alpha=lasso_cv.alpha_)
            lasso.fit(X_mid_train, y_train_real[str(i)])
            y_train_pred=lasso.predict(X_mid_train)
            y_test_pred=lasso.predict(X_mid_test)
            non_zero_coef.append(int(np.sum(lasso.coef_ != 0)))
            if i==3:
                print (non_zero_coef)
              
            result.append(r2_score(y_train_real[str(i)], y_train_pred))
            result.append(r2_score(y_test_real[str(i)], y_test_pred))
        df_result=pd.DataFrame(result,index=indicator_name, columns=[kind+','+method+','+str(latent)])
        df_result_sum=pd.concat([df_result_sum, df_result.T])
        
    return df_result_sum

In [ ]:
df_result_sum=reg(kind="all", method="Lasso", df_result_sum=df_result_sum)
df_result_sum2=reg2(kind="all", method="Lasso", df_result_sum=df_result_sum2)

In [ ]:
df_result_sum=reg(kind="IR", method="Lasso", df_result_sum=df_result_sum)
df_result_sum2=reg2(kind="IR", method="Lasso", df_result_sum=df_result_sum2)

In [ ]:
df_result_sum=reg(kind="WAXS", method="Lasso", df_result_sum=df_result_sum)
df_result_sum2=reg2(kind="WAXS", method="Lasso", df_result_sum=df_result_sum2)

In [ ]:
df_result_sum=reg(kind="POM", method="Lasso", df_result_sum=df_result_sum)
df_result_sum2=reg2(kind="POM", method="Lasso", df_result_sum=df_result_sum2)

In [143]:
df_result_sum.to_csv(path+'/regression/R2_'+model+'_'+str(seed)+'.csv')
df_result_sum2.to_csv(path+'/regression/R2_'+model+'_'+str(seed)+'-2.csv')